In [33]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, roc_auc_score, f1_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed

def load_data(csv_path):
    return pd.read_csv(csv_path)

# Load training data
train_data_path = r"C:\Users\olufe\projects\Journal\dataset\psa_journal_home_A_unsupervise_train.csv"
train_df = load_data(train_data_path)

# Load test data
test_data_path = r"C:\Users\olufe\projects\Journal\dataset\psa_journal_home_A_real_test.csv"
test_df = load_data(test_data_path)

# Assuming the univariate time series data is in the 'value' column
X_train = train_df['Label'].values.reshape(-1, 1)
X_test = test_df['Label'].values.reshape(-1, 1)

# Create a data normalization pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler())
])

# Normalize the data
X_train = pipeline.fit_transform(X_train)
X_test = pipeline.transform(X_test)

def create_lstm_autoencoder(input_shape):
    model = Sequential()
    model.add(LSTM(64, activation='relu', input_shape=input_shape, return_sequences=True))
    model.add(LSTM(32, activation='relu', return_sequences=False))
    model.add(RepeatVector(input_shape[0]))
    model.add(LSTM(32, activation='relu', return_sequences=True))
    model.add(LSTM(64, activation='relu', return_sequences=True))
    model.add(TimeDistributed(Dense(input_shape[1])))
    model.compile(optimizer='adam', loss='mae')
    return model

input_shape = (X_train.shape[1], 1)  # Adjust the input_shape to have only one dimension
model = create_lstm_autoencoder(input_shape)
print(model.summary())

# Rest of the code (training, detecting anomalies, and evaluating the model) remains the same


Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_21 (LSTM)              (None, 1, 64)             16896     
                                                                 
 lstm_22 (LSTM)              (None, 32)                12416     
                                                                 
 repeat_vector_5 (RepeatVect  (None, 1, 32)            0         
 or)                                                             
                                                                 
 lstm_23 (LSTM)              (None, 1, 32)             8320      
                                                                 
 lstm_24 (LSTM)              (None, 1, 64)             24832     
                                                                 
 time_distributed_5 (TimeDis  (None, 1, 1)             65        
 tributed)                                            

In [34]:
#Train the LSTM Autoencoder on the training data:
epochs = 50
batch_size = 32

model.fit(X_train, X_train, epochs=epochs, batch_size=batch_size, validation_split=0.1)


Epoch 1/50
986/986 [==============================] - 14s 8ms/step - loss: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 2/50
986/986 [==============================] - 7s 7ms/step - loss: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 3/50
986/986 [==============================] - 7s 7ms/step - loss: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 4/50
986/986 [==============================] - 8s 8ms/step - loss: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 5/50
986/986 [==============================] - 8s 8ms/step - loss: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 6/50
986/986 [==============================] - 8s 8ms/step - loss: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 7/50
986/986 [==============================] - 8s 9ms/step - loss: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 8/50
986/986 [==============================] - 8s 8ms/step - loss: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 9/50
986/986 [==============================] - 8s 8ms/step - loss: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 10/50
986/986 [======

In [35]:
#Detect anomalies using the Mean Absolute Deviation (MAD) threshold:
def detect_anomalies(data, threshold):
    reconstructions = model.predict(data)
    mse = np.mean(np.power(data - reconstructions, 2), axis=1)
    mad = np.median(mse)
    return mse > threshold * mad

# Calculate the threshold based on the training data
threshold = 3.0  # You can experiment with different values to find the best threshold

# Detect anomalies in the test data
anomalies = detect_anomalies(X_test, threshold)


548/548 [==============================] - 3s 4ms/step


In [36]:
#Evaluate the model using performance metrics:
def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    tnr = tn / (tn + fp)
    fpr = fp / (tn + fp)
    fnr = fn / (fn + tp)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    return accuracy, precision, recall, tnr, fpr, fnr, f1, auc

# Assuming ground truth labels for the test data are available in 'label' column
y_true = test_df['Label'].values

# Convert anomalies to binary labels for evaluation (1: Anomaly, 0: Normal)
y_pred = anomalies.astype(int)

# Calculate performance metrics
accuracy, precision, recall, tnr, fpr, fnr, f1, auc = calculate_metrics(y_true, y_pred)

# Print the performance metrics
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"TNR: {tnr}")
print(f"FPR: {fpr}")
print(f"FNR: {fnr}")
print(f"F1-score: {f1}")
print(f"AUC: {auc}")


Accuracy: 0.5
Precision: 0.0
Recall: 0.0
TNR: 1.0
FPR: 0.0
FNR: 1.0
F1-score: 0.0
AUC: 0.5


C:\Users\olufe\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
